# Integrated Intelligent Systems: Tutorial 3 - Body Motion Problem (Scientific Object) (3 Pts)

This notebook introduces a formulation body motion problem, central to designing and implementing Cognition-Enabled Robot Agents, as a description logic system.

| | |
|---|---|
| **Name** | SHRINIHA PALANIYANDI ASHOKKUMAR |
| **Matriculation no.** | 6570093 |
| **Program** | M.Sc. Artificial Intelligence and Intelligent Systems (AIIS), University of Bremen |
| **Semester** | Winter semester 2026/27 |
| **University email** | shrinih1@uni-bremen.de |
| **GitHub** | [github.com/shriniha](https://github.com/shriniha) |
| **Assignment** | Assignment 3: Body Motion Problem (Tutorial 3) |
| **Deadline** | 24.11.2026, 4 PM |

## Part 1: Robot-World Manipulation System as Scientific Object (E1–E6) (1.5 Pt)

**Title:** Formalizing the Robot Agent: From Architecture to Description Logic Reasoners

![Alternative Text](RWMS.png)

### Scenario Background
An autonomous robotic agent (the PR2 in the EASE apartment) is deployed to execute a manipulation task ("Serve Eva a drink"). To validate this agent scientifically, we must formalize the system as a **Robot-World Manipulation System (RWMS)**, separate the controlling subsystem from the controlled body-world system, define trajectories as trees, and assess runs against an external problem standard using a description logic reasoner.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E1 · Architectural Cut & Subsystem Separation**
   - In the graphical representation of the RWMS, a strict architectural "cut" is placed between the manipulation executive and the body-world system ($[B \bowtie W]$). Explain the nature of this cut: why is the manipulation executive classified as a `dul:Agent` (non-physical program) while the body-world system is classified under physical objects, and how do they interact exclusively across the boundary?

   **ANSWER:**
   The RWMS is split into two subsystems by a strict cut:
   
   - Controlling subsystem: the manipulation executive (the red dashed box). It reads, decides and commits. It is a program made of plans, policies and a control loop. It has no mass, position or contact, so it is non-physical. It is a dul:Agent because it is the entity that decides and acts on the task request (“serve Eva a drink”), not because it is a body.
   
   - Controlled subsystem: the body-world system [B PR2] ⋈ [W kitchen]. It contains the robot body (arms, grippers, camera, base) and the world (fridge, cupboards, container, drink, Eva). All of these are physical objects with states that change under physical law.
   
   Interaction across the cut is possible only through two channels:
   
   - Action tokens (executive → body), such as “position-controlled pull” or “close gripper”.
   
   - Observations (body → executive), such as joint states, force readings and camera images.
   
   The executive has no direct line to the world. It changes the world only through the body, and it learns about the world only through the body’s sensors.
   
   The ⋈ between body and world stands for three couplings inside the physical side: sensing (perceive), contact and force (act), and the world’s own dynamics (change state). The world also evolves by itself, so the executive does not fully determine what happens. The task request enters from outside and belongs to neither subsystem, which matters for E5.


2. **E2 · Forward-Branching Time & World Choices**
   - During an operation on a fridge door or container seal, the same action token (e.g., a position-controlled pull) can lead to completely divergent physical outcomes depending on world choices (e.g., seal releases vs. seal holds tight). Draw or describe the structure of the resulting courses, and explain why the possible courses form a tree that branches *only forward* from the initial state $x_0$.

   **ANSWER:**
   When the same action token is executed at the same state, the world can resolve it in different ways. With a pull on a fridge door:

   ![Alternative Text](e2_forward_branching_tree.png)

   Each node is a physical state. At each step two kinds of choice create branches:
   
   - The agent’s choice of the next action token.- - The world’s choice of outcome (seal releases or holds, handle slips or not).
   
   The set of possible courses forms a tree that branches only forward from x₀ for these reasons:
   
   - Single history: every state has exactly one parent, because it was reached by exactly one sequence of events. Branches never merge.
   - Irreversibility of time: the past is fixed and only the future is open. A torn handle or spilled drink cannot be un-happened.
   - Histories distinguish states: even if two courses reach physically identical states, they are different nodes because their histories differ. This is what makes the structure a tree and not a general graph.

3. **E3 · Trajectory Classification & Priority Hierarchies**
   - The problem standard $P$ sorts every physical trajectory into one of four classes: **SOL** (Solution), **REPD** (Repaired), **NON** (Nonachievement), and **PROH** (Prohibited). Consider two runs that both successfully place the drink within Eva's reach, but where Run A tears off the container handle (`P4` violation) while Run B maintains licensed contact. Classify both runs and explain why the classifier grades the *entire course* rather than just the final end-state.

   **ANSWER:**
   
   Classification of the two runs
   
   | Run | What happens | Class |
   |---|---|---|
   | A | Drink ends within Eva's reach, but the container handle is torn off (`P4` violated) | **PROH** (Prohibited) |
   | B | Drink ends within Eva's reach, contact stays licensed | **SOL** (Solution) |
   
   Meaning of the four classes
   
   - SOL: the goal is reached and no prohibition is violated.
   - REPD: a deviation occurred but was repaired, and the goal was still reached.
   - NON: the goal was never reached (nonachievement).
   - PROH: a prohibition was violated at some point, whatever else happened.
   
   Priority hierarchy: PROH takes precedence over everything else. Violating a prohibition disqualifies a run even if the goal is achieved. Below that, SOL is better than REPD, which is better than NON.
   
   Why the classifier grades the entire course, not just the end-state
   
   1. Prohibitions are constraints on the *path*. The violation in Run A happens mid-course, when the handle is torn off.
   2. The end states of A and B look the same ("drink in reach"), so an end-state check could not tell them apart.
   3. Irreversible damage is not erased by a good final state, and repairs (REPD) are visible only in the sequence of states.
   4. The problem standard `P` is defined over trajectories (courses in the forward-branching tree), not over single states.
   


4. **E4 · Description Logic Partonomy & DUL Constraints**
   - When formalizing the RWMS in description logic under DOLCE Ultra Lite (DUL), we define an RWMS as having *exactly one* executive and *exactly one* body-world system (`RWMS = (=1 hasComponent.ManipExecutive) ⊓ (=1 hasComponent.BodyWorldSystem)`). Using the transitivity of `dul:isPartOf`, explain how a reasoner (such as HermiT) automatically entails that the PR2's head camera is a component of the RWMS without explicitly hardcoding every part-whole link.

   **ANSWER:**
   
   The RWMS is defined by its top-level components:
   
   `RWMS ≡ (=1 hasComponent.ManipExecutive) ⊓ (=1 hasComponent.BodyWorldSystem)`
   
   This fixes exactly one executive and exactly one body-world system. We do **not** list every part under the RWMS. We state only local part-whole links:
   
   - `headCamera ⊑ ∃ isComponentOf.PR2Head`
   - `PR2Head ⊑ ∃ isComponentOf.PR2Body`
   - `PR2Body ⊑ ∃ isComponentOf.BodyWorldSystem`
   - `BodyWorldSystem ⊑ ∃ isComponentOf.RWMS`
   
   In DUL, `isComponentOf` is a sub-property of `dul:isPartOf`, and `dul:isPartOf` is declared **transitive**. A reasoner such as HermiT chains the links:
   
   camera ⊑ part of head ⊑ part of body ⊑ part of body-world system ⊑ part of RWMS
   It therefore entails `headCamera dul:isPartOf RWMS` without any axiom that links the camera to the RWMS directly.
   
   Benefits
   
   - New parts are added with one local axiom, and the reasoner keeps the whole partonomy consistent.
   - The cardinality axioms tell the reasoner the camera sits under the one and only body-world system, not under a second one.
   

5. **E5 · Ontological Inconsistency & Entailment Limits**
   - An inconsistency in description logic serves as a strict analytical tool. Formulate a competency question that results in an ontological inconsistency when attempting to classify the *manipulation problem itself* as a physical component inside the RWMS. Furthermore, explain why numerical physical laws (such as Coulomb friction $2 \cdot \mu \cdot F \ge m \cdot (g + a)$) must be handled outside description logic by the semantic digital twin's causal model.

   **ANSWER:**
   
   Competency question
   
   > "Can the manipulation problem `P` be a component of the body-world system inside the RWMS?"
   
   Axioms involved
   
   - `BodyWorldSystem ⊑ dul:PhysicalObject`
   - `ManipulationProblem ⊑ dul:Description` (a specification or norm, not a thing)
   - `dul:PhysicalObject ⊓ dul:Description ⊑ ⊥` (disjoint in DUL)
   - Assertion: `P isComponentOf bws`
   
   For `P` to be a component of a physical system it must be a physical object. It is also a Description. These classes are disjoint, so the ontology is **inconsistent**.
   
   This inconsistency is a useful analytical result: the problem standard is *external* to the system it judges. In the architecture figure, the task request enters from outside the controlling subsystem and belongs to neither subsystem.
   
   Why numerical physical laws must be handled outside DL
   
   - OWL/DL expresses classes, roles and simple datatype restrictions. It cannot relate several numeric variables in an arithmetic formula such as $2 \cdot \mu \cdot F \ge m \cdot (g + a)$.
   - Nonlinear real arithmetic would make reasoning undecidable or intractable, which would break guaranteed reasoning in HermiT.
   - Such laws belong to the **semantic digital twin's causal model** (a physics simulation and numerical computation). It evaluates the inequality for the actual values and returns a qualitative result to the knowledge base, for example `grasp1 a SlipFree`. DL then reasons over that result.
   
   This is a division of labour: numbers and dynamics in the causal model, concepts and entailments in DL.

6. **E6 · Strength vs. Truth in Agent Validation**
   - The design thesis dictates that an agent decides within its believed causality, which can be tested for logical *strength* via competency queries and for *truth* via physical experiments. Compare how an axiom set that lacks a friction law ($\text{K}_0$) performs on competency queries versus an axiomatization containing explicit dynamical laws ($\text{K}_1/\text{K}_2$), and state what kind of test is required to validate the actual physical truth of those beliefs.

   **ANSWER:**
   
   (Assumption: $\text{K}_0$ has no friction law, and $\text{K}_1/\text{K}_2$ add friction and further dynamical laws.)
   
   Design thesis: the agent decides within its *believed* causality. Those beliefs can be tested in two different ways.
   
   | Test | Question | Method |
   |---|---|---|
   | Strength | How much do the beliefs entail? | Competency queries to a reasoner |
   | Truth | Do the beliefs match the real world? | Physical experiments |
   
   $\text{K}_0$ (no friction law)
   
   - Under the open-world assumption, queries such as "does the drink stay in the gripper?" or "will this pull yield SOL?" are neither entailed nor refuted.
   - Most competency queries stay unanswered. $\text{K}_0$ is logically weak. It is not wrong; it says too little to support decisions.
   
   $\text{K}_1/\text{K}_2$ (explicit dynamical laws)
   
   - The friction and dynamics axioms let the reasoner entail outcomes, for example "if grip force exceeds the friction threshold, the object holds".
   - Many more competency queries are answered. The agent can predict SOL, NON or PROH before acting and can rule out plans that would tear the handle.
   - $\text{K}_2$ gives more strength than $\text{K}_1$ if it adds further laws.
   
   Strength is not truth. A stronger axiom set can entail many things and still be false. For example, a wrong friction coefficient gives confident but wrong predictions, and competency queries cannot detect that because they only check entailment *inside* the model.
   
   What is needed to validate truth: physical experiments. 
   Run the real PR2, or at least a validated high-fidelity simulation, over many trajectories. Compare the predicted outcome class (SOL, REPD, NON, PROH) with the observed one under `P`. Mismatches falsify or refine the belief set, and agreement across varied conditions builds empirical confidence.

## Part 2: Robot Motion Problem and Description Logics (1.5 Pt)

**Title:** Formulating and Querying the Robot Motion Problem with Knowledge Bases & Description Logics

![Alternative Text](PR2.png)

### Scenario Background
Using the interactive Jupyter notebook and RViz simulation environment (`bmp.rviz`, `setup_demo()`), the PR2 mobile manipulator interacts with the EASE apartment model (including the fridge and various kitchen drawers). Instead of hardcoding trajectories, the robotic executive queries a knowledge base (represented via Description Logics / Prolog reasoning) to link task requests to state changes, causes (motions), and capabilities.

[Go to the programming Lab](https://binder.intel4coro.de/v2/gh/maltehue/giskard-examples/bmp_vrb?urlpath=/lab/tree/notebooks/bodyMotionProblem.ipynb)
---

### Multi-Question Student Task Sheet (0.25 Pt x 6)

1. **E7 · Knowledge Base Grounding & Task Requests**
   - Execute the initial query `taskRequest(Request, StateChange).`Examine the resulting solution bindings (e.g., `{'Request': 'open_fridge', 'StateChange': 'open(fridge)'}` and various drawer operations). Explain how mapping abstract task requests to discrete state changes forms the top-level specification layer for the manipulation executive.

   **ANSWER:**
   **Query**
   
   ```prolog
   taskRequest(Request, StateChange).
   ```
   
   **Result (17 solutions, then "No more solutions")**
   
   | Request | State changes |
   |---|---|
   | `open_fridge` | `open(fridge)` |
   | `open_anything` | `open(drawer_01)` … `open(drawer_06)`, `open(oven_drawer)`, `open(fridge)` |
   | `close_anything` | `close(drawer_01)` … `close(drawer_06)`, `close(oven_drawer)`, `close(fridge)` |
   
   **Explanation**
   
   - `taskRequest/2` maps an abstract, human-level request ("open the fridge") to a discrete **state change** of the environment (`open(fridge)`). The request says *what the user wants*, and the state change says *what must be true of the world afterwards*.
   - One request can map to many state changes. `open_anything` ranges over all 8 openable objects, while `open_fridge` is a specific request with exactly one state change.
   - This mapping is the **top-level specification layer** for the manipulation executive. The executive is not given a trajectory. It is given a goal state change, and everything below (which motion, which parameters, which robot) is derived from it.
   - The mapping is declarative and lives in the knowledge base. New tasks are added as facts, and the executive does not change.
   - Each state change also gives the success criterion that the problem standard $P$ can use when classifying a run (SOL, REPD, NON, PROH).
   - This is the first of the three steps of the law of task-achieving body motion: **request → state change**, then **state change → motion** (E8), then **motion → capable robot** (E9).

2. **E8 · Extending Queries for Causal Motions & Parameters**
   - Extend the query to incorporate motion causes: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam).` Explain how the reasoner bridges the gap between a desired symbolic state change and continuous motion parameters (such as `envJointGoal`, target joint angles, and handle identifiers).

   **ANSWER:**

   **Query**
   
   ```prolog
   taskRequest(Request, StateChange),
   causes(Motion, StateChange, MotionParam).
   ```
   
   **Result (9 solutions)**, for example:
   
   ```
   {'Request': 'open_fridge', 'StateChange': 'open(fridge)', 'Motion': 'envJointGoal',
    'MotionParam': [0.8, b'fridge_door_joint', b'fridge_door_handle']}
   {'Request': 'open_anything', 'StateChange': 'open(drawer_03)', 'Motion': 'envJointGoal',
   'MotionParam': [0.2, b'drawer_03_joint', b'drawer_03_handle']}
   ```
   
   **Explanation**
   
   - `causes/3` states which **motion** brings about a given state change, and with which **parameters**. It links the symbolic level (`open(fridge)`) to the control level.
   - Prolog unifies the shared variable `StateChange`. The state change returned by `taskRequest` is used to look up the motion that causes it.
   - Every solution uses the same motion type, `envJointGoal`: move an *environment joint* to a goal value. The `MotionParam` list carries what the controller needs:
    - the **target joint goal** (`0.8` for the fridge door, `0.2` for each drawer, presumably radians for the revolute door and metres for the prismatic drawers),
    - the **joint identifier** (e.g. `fridge_door_joint`, `drawer_03_joint`),
    - the **handle identifier** (e.g. `fridge_door_handle`), which is where the gripper must make contact.\
   - The result drops from 17 to **9** solutions. All 8 `close_anything` state changes disappear, because the knowledge base has no `causes` entry for them. The reasoner can only return a motion if one is defined for that state change.
   - So the reasoner bridges the gap by *retrieval and unification*. It does not compute trajectories. It selects the motion type and fills in its parameters from the knowledge base.

3. **E9 · Integrating Robot Capabilities (`canPerform`)**
   - Finalize the query by adding the robot capability constraint: `taskRequest(Request, StateChange), causes(Motion, StateChange, MotionParam), canPerform(Robot, Motion, MotionParam).`Describe what role the `Robot` variable (`'pr2'`) plays in filtering feasible trajectories through the reasoner.

   **ANSWER:**

   **Query**
   
   ```prolog
   taskRequest(Request, StateChange),
   causes(Motion, StateChange, MotionParam),
   canPerform(Robot, Motion, MotionParam).
   ```
   
   **Result:** 18 solutions, each with `'Robot': 'pr2'`. Each of the 9 request/motion combinations from E8 appears twice, for example:
   
   ```
   {'Request': 'open_fridge', 'StateChange': 'open(fridge)', 'Motion': 'envJointGoal',
   'MotionParam': [0.8, b'fridge_door_joint', b'fridge_door_handle'], 'Robot': 'pr2'}
   ```
   
   **Role of the `Robot` variable**
   
   - `Robot` is left **unbound** in the query. Prolog searches for a robot that can perform the motion, and the solutions come back with `Robot = 'pr2'`. So `'pr2'` is the answer the reasoner finds. You can also bind it yourself (`canPerform(pr2, Motion, MotionParam)`) to ask "can *this* robot do it?".
   - `canPerform/3` is a **capability constraint**. It succeeds only if the given robot can execute the given motion with the given parameters. It ties the abstract task to a concrete **embodiment**.
   - It works as a **filter** on the solution set. Motions that are correct for a state change but not feasible for the robot would be removed because `canPerform` fails for them. Here the PR2 can perform `envJointGoal` for every state change that has a cause, so all 9 combinations survive.
   - This is the third step of the law: request → state change → causing motion → **a robot that can perform it**. A solution of the full query is a motion the PR2 can actually carry out.
   - Each solution appears twice. The most likely reason is that the knowledge base can derive `canPerform(pr2, ...)` in two ways (for example two matching rules or facts). Prolog returns each derivation separately, and the pairs of identical rows are not different solutions.

4. **E10 · Specializing Request Scope (`open_fridge` vs. `open_anything`)**
   - Modify the query request parameter from general categories like `open_anything` to specifically target `open_fridge`. Explain how refining the scope in the knowledge base reduces the solution search tree and prevents unauthorized or unintended state changes in the kitchen environment.

   **ANSWER:**

   **Modified query**
   
   ```prolog
   taskRequest(open_fridge, StateChange),
   causes(Motion, StateChange, MotionParam),
   canPerform(Robot, Motion, MotionParam).
   ```
   
   (In the lab: the third query with `Request` replaced by `open_fridge`.)
   
   **Result:** only the fridge solution remains, `open(fridge)` with `[0.8, b'fridge_door_joint', b'fridge_door_handle']` and `Robot = 'pr2'`, shown twice because of the duplicate derivation described in E9. This is **2 rows instead of 18**.
   
   **Explanation**
   
   - With `open_anything`, `Request` unifies with every openable object. Prolog explores all 8 branches (the 7 drawers/oven plus the fridge), giving many solutions.
   - Binding `Request` to `open_fridge` **prunes the search tree at the start**. Only the single `taskRequest` fact for the fridge matches, so only `open(fridge)` is passed on to `causes/3` and `canPerform/3`.
   - This gives a smaller and faster search and a single, unambiguous goal for the executive.
   - It also keeps the robot's behaviour within the **authorised scope**. A request for the fridge cannot produce a drawer or oven motion, so no unintended state change in the kitchen is derived or executed.
   - In terms of the problem standard $P$, a narrower request reduces the risk that a run is classified as PROH because of side effects on other objects.

5. **E11 · Visualization and Simulation Linkage in RViz**
   - With the background shell command executing `rviz -d .../bmp.rviz` and displaying the PR2 interacting with the fridge handle, explain how the motion parameters derived from the reasoner's output (`[0.8, b'fridge_door_joint', b'fridge_door_handle']`) are executed as physical joint trajectories in the simulation.

   **Motion parameters from the reasoner:** `[0.8, b'fridge_door_joint', b'fridge_door_handle']`, with motion `envJointGoal` and `Robot = 'pr2'`.
   
   **How they become motion in the simulation**
   
   1. **Reasoner output → motion goal.** The executive takes the tuple as arguments of a motion command: *move the environment joint `fridge_door_joint` to position `0.8`, with the gripper holding `fridge_door_handle`*.
   2. **Motion goal → controller.** The goal is handed to the motion controller (in this lab most likely the Giskard whole-body motion framework, judging from the repository name `giskard-examples`). It turns the goal into constraints: the gripper must hold the handle, and the door joint must reach the target value.
   3. **Controller → joint trajectories.** The controller solves for the PR2's joint commands over time (arm, torso and base). These are the **physical joint trajectories** executed in the simulation.
   4. **Simulation → RViz.** The simulated robot and environment publish their joint states and transforms. RViz, started in the background with `rviz -d .../bmp.rviz`, renders them in the right window. `setup_demo()` connects the notebook queries to this simulation. So you see the PR2 reach the fridge handle and the door swing open.
   
   **Key point:** the reasoner supplies *what* (symbolic goal plus parameters), and the controller and simulation supply *how* (continuous motion). The two are linked only through the motion parameters.

6. **E12 · Ontological Limits & Missing Dynamics**
   - While the knowledge base successfully infers *which* motion achieves a state change, explain why a pure Description Logic or symbolic Prolog reasoner cannot verify whether a specific grasp force will cause a wet carton to slip, and identify which subsystem must handle those physical calculations.

   **ANSWER:**
   
   - The knowledge base can infer *which* motion achieves a state change (`envJointGoal` for `open(fridge)`), because that is symbolic retrieval and unification over facts and rules. The results above show exactly that: motion type, goal value and identifiers, and nothing about forces.
   - Whether a carton slips depends on **continuous physical quantities**: grip force $F$, friction coefficient $\mu$ (lower when the carton is wet), mass $m$, gravity $g$ and acceleration $a$. The condition is a numerical inequality, e.g. $2 \cdot \mu \cdot F \ge m \cdot (g + a)$.
   - DL and Prolog do not evaluate this kind of arithmetic over several real-valued variables in a decidable, guaranteed way. They can only reason over *given* qualitative facts such as "grasp is SlipFree". They cannot compute such facts from forces.
   - Without a friction law in the axioms, the question is simply unanswerable (open world). Even with a qualitative friction rule, the actual coefficient and force values must come from outside the knowledge base.
   
   **Subsystem responsible:** the **physical (causal) model of the semantic digital twin**, i.e. the physics simulation together with the controller and force/torque sensing of the body-world system. It evaluates the dynamics and returns a qualitative result to the knowledge base. Whether that model is *true* is then tested by physical experiments.



# Submission guidelines

- Deadline: 24.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (If not yet, create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put the textual answers in notebook and all related media (images, rules, etc) should then be put into a folder called **Assignment3**. 